# **뉴스 카테고리 분류 파인튜닝**

## 1.환경준비

### (1) 라이브러리 설치 및 로딩

In [ ]:
!pip install datasets -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch

from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from datasets import load_dataset, Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import *
from sklearn.preprocessing import LabelEncoder

from warnings import filterwarnings
FutureWarning
filterwarnings('ignore')

* 함수 생성

In [ ]:
# 검증셋 평가 함수
def evaluate(val_ds, model, device, tokenizer):
    # 입력 데이터셋 토크나이징
    texts = list(val_ds['text'])
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)
    inputs = {key: value.to(device) for key, value in inputs.items()}  # 입력 텐서를 동일한 디바이스로 이동

    # 모델을 지정된 디바이스로 이동
    model = model.to(device)
    model.eval()
    with torch.no_grad():  # 평가 과정에서 기울기 계산 비활성화
        outputs = model(**inputs)  # attention_mask를 포함해 입력

    # 예측 및 확률 계산
    prob = outputs.logits.softmax(dim=1)

    # prob가 GPU에 있을 경우에만 CPU로 이동
    if prob.is_cuda:
        prob = prob.cpu().detach().numpy()
    else:
        prob = prob.detach().numpy()

    # 가장 확률이 높은 위치 값(인덱스) 반환
    pred = np.argmax(prob, axis=1)

    # GPU 메모리에서 필요 없는 텐서 제거 및 캐시 정리
    del inputs
    torch.cuda.empty_cache()

    return pred, prob

In [ ]:
def predict(text, model, tokenizer):
    # 모델을 CPU로 이동
    model = model.to("cpu")

    # 입력 문장 토크나이징 → CPU 텐서로 생성
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    # 모델 예측 (no_grad로 메모리 절약)
    with torch.no_grad():
        outputs = model(**inputs)

    # 확률 계산
    logits = outputs.logits
    prob = torch.softmax(logits, dim=1)

    # 예측 클래스
    pred = torch.argmax(prob, dim=-1).item()

    return pred, prob


### (2) 데이터 로딩

In [ ]:
data = pd.read_csv('https://raw.githubusercontent.com/DA4BAM/dataset/refs/heads/master/naver_news_title.csv')
data.rename(columns = {'titles' : 'text', 'category' : 'label'}, inplace = True)
data.drop_duplicates(subset=['text'], inplace=True)
data = data.sample(5000)
data.head()

* y 분포 확인하기

In [ ]:
sns.countplot(x='label', data = data, palette='Set2')
plt.grid()
plt.show()

* label 에 대한 정수 인코딩

In [ ]:
le = LabelEncoder()
data['label'] = le.fit_transform(data['label'])
data.head()

In [ ]:
label_list = list(le.classes_)
label_list

### (3) GPU 설정
* 파이토치를 위한 설정

In [ ]:
# GPU 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## 2.데이터 준비

### (1) Dataset 만들기 : train, val

In [ ]:
train, val = train_test_split(data, test_size=0.3, random_state=100)

In [ ]:
# df로 부터 텐서 데이터셋 만들기
train_ts = Dataset.from_pandas(train)
val_ts = Dataset.from_pandas(val)

In [ ]:
train_ts[:3]

### (2) 토크나이징

In [ ]:
# 모델과 토크나이저 불러오기
model_name = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
# 토큰화 함수 생성 및 작업
def preprocess_function(data):
    return tokenizer(data['text'], truncation=True, padding=True)

train_ts = train_ts.map(preprocess_function, batched=True)
val_ts = val_ts.map(preprocess_function, batched=True)

In [ ]:
train_ts[0]

## 3.Fine-Tuning

### (1) 사전학습 모델 준비

In [ ]:
# 1. 모델 로드
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=6)

### (2) 학습 설정

In [ ]:
training_args = TrainingArguments(
    output_dir = './results',
    eval_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate = 2e-5,               # 작은 학습률
    per_device_train_batch_size = 32,   # 학습 배치 사이즈
    per_device_eval_batch_size = 32,
    num_train_epochs = 5,               # 에폭 수
    weight_decay = 0.02,                # weight decay
    load_best_model_at_end = True,      # earlystopping 사용하기 위해 필요
    logging_dir ='./logs',
    logging_steps = 10,
    report_to="tensorboard"
)

In [ ]:
# Trainer 설정
trainer = Trainer(
    model = model,                         # 학습할 모델
    args=training_args,                  # TrainingArguments
    train_dataset = train_ts,
    eval_dataset = val_ts,
    tokenizer = tokenizer,
    callbacks = [EarlyStoppingCallback(early_stopping_patience=3)], # 조기 종료
)

### (3) 학습
* 위 코드 그대로

In [ ]:
# 모델 학습
trainer.train()

### (4) 모델 사용

In [ ]:
text = "파인튜닝 모델을 이용해서 저비용 고성능 모델 개발"
pred, prob = predict(text, model, tokenizer)

print(f"예측된 클래스: {pred}")
print(f"예측된 클래스 이름: {label_list[pred]}")
print(f"클래스별 확률: {prob}")

### (5) 모델 검증평가

In [ ]:
pred, prob = evaluate(val_ts, model, device, tokenizer)

In [ ]:
cm = confusion_matrix(val_ts['label'], pred)

disp = ConfusionMatrixDisplay(cm, display_labels = label_list)
disp.plot()
plt.show()

print(classification_report(val_ts['label'], pred, target_names = label_list))